In [1]:
import re
import torch
from torch.utils import data
def load_text_data(
    file_path,
    batch_size,
    num_steps,
    use_random_iter=True,
    max_tokens=None
):
    # 1. 读取并简单清洗文本
    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as file:
        text = file.read().lower()

    # 将换行、连续空格统一成一个空格
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    if max_tokens is not None:
        text = text[:max_tokens]

    # 2. 字符级词表
    tokens = list(text)

    idx_to_token = [
        "<unk>"
    ] + sorted(set(tokens))

    token_to_idx = {
        token: index
        for index, token
        in enumerate(idx_to_token)
    }

    # 3. 文本转换为编号
    corpus = torch.tensor(
        [
            token_to_idx[token]
            for token in tokens
        ],
        dtype=torch.long
    )

    if use_random_iter:
        # ====================================================
        # 随机抽样
        # 每一行包含num_steps + 1个token
        # ====================================================

        sequences = corpus.unfold(
            dimension=0,
            size=num_steps + 1,
            step=num_steps
        )

        X = sequences[:, :-1]
        Y = sequences[:, 1:]

        dataset = data.TensorDataset(
            X,
            Y
        )

        train_iter = data.DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True
        )

    else:
        # ====================================================
        # 顺序抽样
        # 保证相邻batch在原始文本中连续
        # ====================================================

        num_tokens = (
            (len(corpus) - 1)
            // batch_size
            * batch_size
        )

        X_all = corpus[
            :num_tokens
        ].reshape(
            batch_size,
            -1
        )

        Y_all = corpus[
            1:num_tokens + 1
        ].reshape(
            batch_size,
            -1
        )

        num_batches = (
            X_all.shape[1]
            // num_steps
        )

        usable_length = (
            num_batches
            * num_steps
        )

        # 形状变为：
        # (num_batches, batch_size, num_steps)
        X = X_all[
            :, :usable_length
        ].reshape(
            batch_size,
            num_batches,
            num_steps
        ).permute(1, 0, 2)

        Y = Y_all[
            :, :usable_length
        ].reshape(
            batch_size,
            num_batches,
            num_steps
        ).permute(1, 0, 2)

        dataset = data.TensorDataset(
            X,
            Y
        )

        # Dataset中的一条数据本身已经是一个batch
        train_iter = data.DataLoader(
            dataset,
            batch_size=None,
            shuffle=False
        )

    vocab = {
        "token_to_idx": token_to_idx,
        "idx_to_token": idx_to_token
    }

    return train_iter, vocab

In [2]:
batch_size = 32
num_steps = 35

train_iter, vocab = load_text_data(
    file_path="../data/timemachine.txt",
    batch_size=batch_size,
    num_steps=num_steps,
    use_random_iter=True,
    max_tokens=10000
)

for X, Y in train_iter:
    print("X形状：", X.shape)
    print("Y形状：", Y.shape)

    print("第一条X：", X[0])
    print("第一条Y：", Y[0])

    break

X形状： torch.Size([32, 35])
Y形状： torch.Size([32, 35])
第一条X： tensor([20, 22, 33, 37, 22, 21,  1, 18, 20, 20, 32, 38, 31, 37,  1, 32, 23,  1,
        37, 25, 22,  1, 19, 18, 37, 37, 29, 22,  1, 32, 23,  1, 25, 18, 36])
第一条Y： tensor([22, 33, 37, 22, 21,  1, 18, 20, 20, 32, 38, 31, 37,  1, 32, 23,  1, 37,
        25, 22,  1, 19, 18, 37, 37, 29, 22,  1, 32, 23,  1, 25, 18, 36, 37])


In [3]:
x_text = "".join(
    vocab["idx_to_token"][index]
    for index in X[0].tolist()
)

y_text = "".join(
    vocab["idx_to_token"][index]
    for index in Y[0].tolist()
)

print("X：", x_text)
print("Y：", y_text)

X： cepted account of the battle of has
Y： epted account of the battle of hast


In [4]:
def my_corpus_vocab_maker(max_token=None):
    with open("../data/timemachine.txt",'r') as file:
        text=file.read().lower()
    #print(text)
    text=re.sub("[^A-Za-z]+",' ',text).strip().lower()
    #print(text)
    #token=list(text)
    tokens=text.split()
    idx_to_token=[*(['<unk>']+list(set(tokens)))]
    token_to_idx={
        token:idx
        for idx,token in enumerate(idx_to_token)
    }
    corpus=torch.tensor([token_to_idx[token] for token in tokens])
    vocab={
        "idx_to_token":idx_to_token,
        "token_to_idx":token_to_idx

    }
    return corpus,vocab
corpus,vocab=my_corpus_vocab_maker()
print(torch.arange(30))
print(corpus[:20])
print(
    [vocab["idx_to_token"][idx] for idx in corpus[:20].tolist()]
)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])
tensor([3401, 1444, 4505, 1292,  243, 4123, 3294, 2508, 3401, 1444,  839, 2175,
         975, 3836, 2991, 2357, 2962, 2162, 1409,  484])
['the', 'time', 'machine', 'by', 'h', 'g', 'wells', 'i', 'the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of']


In [5]:
from torch.utils import data
def train_iter_maker(
    corpus: torch.Tensor,
    batch_size: int,
    num_steps: int
) -> data.DataLoader:

    corpus = corpus.flatten().long()

    if len(corpus) <= num_steps:
        raise ValueError(
            "corpus长度必须大于num_steps"
        )

    # 每段包含num_steps+1个token
    sequences = corpus.unfold(
        dimension=0,
        size=num_steps + 1,
        step=num_steps
    )

    # 输入和标签错开一个位置
    X = sequences[:, :-1].contiguous()
    Y = sequences[:, 1:].contiguous()

    dataset = data.TensorDataset(
        X,
        Y
    )

    train_iter = data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    return train_iter

In [6]:
train_iter=train_iter_maker(corpus,2,4)